[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

*Training*
Implement **cross-entropy loss directly from logits** — the plain form, no extras.

$$\ell_i = -\log p_{i,t_i}, \qquad
p_{i,c} = \frac{e^{z_{i,c}}}{\sum_{k} e^{z_{i,k}}}$$

Return the **mean over the batch**.

### Rules
- Signature: `cross_entropy_loss(logits, targets)`
- `logits` is `(B, C)`, `targets` is `(B,)` of integer class ids; the output is a **scalar**
- Banned: `jax.nn.log_softmax`, `jax.nn.softmax`, `jax.scipy.special.logsumexp`, `optax`
- Compute $\log p$ in one fused expression; never form $p$ and then take its log
- It has to survive extreme logits, and work under `jit`

### Why you never softmax-then-log
The naive route dies twice.

**Overflow.** `exp(z)` is `inf` above $z \approx 88.7$ in float32 and above
$z \approx 11.1$ in float16. The fix is the shift
$\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$ with $m = \max_k z_k$: every
exponent is now $\le 0$, so the largest term is exactly `1.0` and the sum can
never overflow.

**Underflow — the one that actually bites.** Even with no overflow, a confidently
*wrong* prediction pushes $p_t$ under the float32 floor: normals stop at
$\approx 1.2\times10^{-38}$ and subnormals at $\approx 1.4\times10^{-45}$, and
XLA flushes subnormals to zero on accelerators anyway. Once $p_t$ rounds to
`0.0`, `log(0) = -inf` makes the loss `inf` and every gradient `nan`. The fused
form never materialises $p_t$: it computes $z_t - \log\sum_k e^{z_k}$, a
perfectly finite number like $-120$ (that is $p_t \approx 10^{-52}$, hopelessly
unrepresentable, yet its logarithm is an ordinary float). Your loss stays
large-but-finite and training recovers instead of poisoning every parameter
with `nan`.

There is a gradient bonus too. $\partial \ell / \partial z = p - q$ — a clean,
bounded expression that autodiff derives exactly from the fused form. Compose
`log` on top of a separate `softmax` and you hand XLA a division of two tiny
numbers to differentiate through.

The `max` shift cancels analytically ($\ell$ is invariant to it), so wrapping it
in `stop_gradient` changes nothing mathematically and keeps the backward graph
smaller.

### Gathering the target
$-\log p_{i,t_i}$ needs one entry per row. `jnp.take_along_axis(log_probs,
targets[:, None], axis=-1)` reads exactly those and nothing else. A one-hot
matmul gets the same answer by building a `(B, C)` array of zeros to multiply
against — correct, but it is `B×C` work and memory for `B` numbers, which at
vocabulary sizes is the difference between a loss that fits and one that does
not.

Once this passes, **`b_14` (Cross-Entropy: Smoothing & Padding Mask)** picks it
up and adds the two arguments real training code always passes.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cross_entropy_loss(logits, targets):
    """Mean cross-entropy over the batch.

    Args:
        logits:  (B, C) unnormalised scores
        targets: (B,) integer class ids

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# Uniform logits over 3 classes -> loss is exactly log(3).
print("uniform:", cross_entropy_loss(jnp.zeros((1, 3)), jnp.array([0])), "vs", jnp.log(3.0))

# Random data against the library log-softmax.
logits = jax.random.normal(jax.random.key(0), (4, 5)) * 3.0
targets = jnp.array([1, 2, 0, 4])
ref = -jnp.mean(jnp.take_along_axis(
    jax.nn.log_softmax(logits, axis=-1), targets[:, None], axis=-1))
print("mine:", float(cross_entropy_loss(logits, targets)), " ref:", float(ref))

# The stability trap: huge logits.
big = jnp.array([[1000.0, 0.0, 0.0]])
print("\nbig logits, correct class:", cross_entropy_loss(big, jnp.array([0])))
print("naive softmax-then-log would give:",
      -jnp.log(jnp.exp(big) / jnp.exp(big).sum(-1, keepdims=True))[0, 0])

# Confidently WRONG stays finite — the loss is ~1000, not inf.
print("big logits, wrong class:  ", float(cross_entropy_loss(big, jnp.array([1]))))

# The gradient is p - onehot, averaged over the batch.
g = jax.grad(cross_entropy_loss)(jnp.array([[2.0, 1.0, 0.0]]), jnp.array([0]))
print("\ngrad:", g, " sums to", float(jnp.sum(g)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_entropy")

# hint("cross_entropy")      # stuck? nudge without the answer
# solution("cross_entropy")  # spoiler: the reference implementation
# status()                   # your dashboard across all problems